## Kintsugi Health — DAM-like Whisper fine-tune (RADAR, 4 sites)

Adapted from the original Androids HC/PT notebook. Fine-tunes a **frozen Whisper encoder + trainable MLP head** on binary depression labels (**PHQ-8 >= 10**), with a **grouped train/test split by `participant_id`** so no participant appears in both splits.

**RADAR data**: loads all 4 site CSVs (`radar_model_dataset_raw_features-KCL.csv`, `-VUmc.csv`, `-CIBER.csv`, `-IISPV.csv`) from `data/processed/`, reusing the same column-standardising + audio-resolution logic built for `RADAR_Embedding_Extraction.ipynb` (composite `(participant_id, filename)` lookup — avoids the cross-participant filename collisions found earlier). `RADAR_AUDIO_ROOT` points at the OneDrive folder containing the `.wav` files.

**What changed from the Androids version**: no more HC/PT folders — `subgroup` now holds the **site** (KCL/VUmc/CIBER/IISPV) instead of HC/PT; `participant_id` comes from RADAR metadata instead of being parsed from the filename; `depressed` is derived from `phq8_score >= 10` instead of the folder label; device selection now checks Apple Silicon `mps` first (previous CUDA-only check would always fall back to CPU on this machine); Whisper padding uses `padding="max_length"` (`padding=True` throws a mel-length mismatch — hit and fixed during the embedding-extraction notebook work).

**Before running**: confirm `RADAR_AUDIO_ROOT` below still points at your OneDrive RADAR-MDD folder, and that `BATCH_SIZE`/`EPOCHS` are reasonable for your hardware — RADAR (~13.9k recordings) is far larger than the Androids HC/PT set this notebook was originally written for.

In [6]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

# Hardcoded to match RADAR_Embedding_Extraction.ipynb -- change if this repo
# lives somewhere else on your machine.
PROJECT = Path("/Users/k1777551/JansCode/ASMHI-Research-Project")

# RADAR metadata/PHQ-8 CSVs: one file per site, already standardised to
# File / participant_id / phq8_score, with PHQ-8 scores matched to each
# recording within a 7-day window (see Merge-PHQ-8.ipynb for how these were built).
RADAR_CSV_DIR = PROJECT / "data" / "processed"
RADAR_CSV_PATTERN = "radar_model_dataset_raw_features-*.csv"

# RADAR .wav files live on OneDrive:
# RADAR_AUDIO_ROOT/RADAR-MDD-Speech/<site>/<participant_id>/<file>.wav
RADAR_AUDIO_ROOT = Path(
    "/Users/k1777551/Library/CloudStorage/OneDrive-King'sCollegeLondon/RADAR-MDD"
)

# Each site's raw CSV uses slightly different column names for the same fields.
SITE_COLUMN_RENAMES = {
    "KCL": {},
    "CIBER": {"Date": "recording_date", "PHQ8": "phq8_score"},
    "IISPV": {"Date": "recording_date", "PHQ8": "phq8_score"},
    "VUmc": {"Date": "recording_date", "PHQ8": "phq8_score"},
}
REQUIRED_COLUMNS = ["File", "participant_id", "phq8_score"]


def build_wav_filename_index(audio_root: Path) -> dict[tuple[str, str], Path]:
    """Index every .wav under audio_root, keyed by (participant_folder, filename).
    Scans site-by-site with a progress bar since walking the full OneDrive
    tree can take a while."""
    if not audio_root.exists():
        return {}

    speech_root = audio_root / "RADAR-MDD-Speech"
    site_dirs = sorted(d for d in speech_root.iterdir() if d.is_dir()) if speech_root.is_dir() else []

    idx: dict[tuple[str, str], Path] = {}

    if not site_dirs:
        for p in tqdm(list(audio_root.rglob("*.wav")), desc="Indexing all wav files"):
            participant_folder = p.parent.name
            idx[(participant_folder, p.name)] = p
            idx[(participant_folder, p.name.lower())] = p
        return idx

    for site_dir in site_dirs:
        wav_files = list(site_dir.rglob("*.wav"))
        for p in tqdm(wav_files, desc=f"Indexing {site_dir.name}"):
            participant_folder = p.parent.name
            idx[(participant_folder, p.name)] = p
            idx[(participant_folder, p.name.lower())] = p
        print(f"  {site_dir.name}: indexed {len(wav_files)} wav files")

    return idx


def load_and_merge_radar_csvs(csv_dir: Path, pattern: str) -> pd.DataFrame:
    csv_paths = sorted(csv_dir.glob(pattern))
    if not csv_paths:
        print(f"RADAR: no CSV files found matching '{pattern}' in {csv_dir}")
        return pd.DataFrame()

    frames = []
    for p in csv_paths:
        site_key = p.stem.split("-")[-1]
        raw = pd.read_csv(p)
        renamed = raw.rename(columns=SITE_COLUMN_RENAMES.get(site_key, {}))
        missing = [c for c in REQUIRED_COLUMNS if c not in renamed.columns]
        if missing:
            print(f"WARNING: {p.name} (site={site_key}) missing {missing} after rename")
        renamed["source_site_csv"] = site_key
        frames.append(renamed)
        print(f"{p.name}: loaded {len(raw)} rows (site_key='{site_key}')")

    return pd.concat(frames, ignore_index=True, sort=False)


def prepare_radar_audio_df(wav_index: dict[tuple[str, str], Path]) -> pd.DataFrame:
    df = load_and_merge_radar_csvs(RADAR_CSV_DIR, RADAR_CSV_PATTERN)
    if df.empty:
        raise FileNotFoundError(f"No RADAR CSVs found in {RADAR_CSV_DIR}")

    df["participant_id"] = df["participant_id"].astype(str).str.strip()
    df["phq8_score"] = pd.to_numeric(df["phq8_score"], errors="coerce")
    df = df.dropna(subset=["phq8_score", "File", "participant_id"]).copy()
    df["depressed"] = (df["phq8_score"] >= 10).astype(int)

    names = df["File"].astype(str).str.strip()

    def resolve(participant_id: str, name: str):
        if not name:
            return None
        key = (participant_id, name)
        if key in wav_index:
            return wav_index[key]
        return wav_index.get((participant_id, name.lower()))

    audio_paths = df.progress_apply(lambda row: resolve(row["participant_id"], str(row["File"]).strip()), axis=1)
    n_ok = audio_paths.notna().sum()
    if n_ok == 0:
        raise FileNotFoundError(
            "RADAR: could not resolve any WAV paths. Check RADAR_AUDIO_ROOT points at "
            "the OneDrive folder containing the recordings referenced in column 'File'."
        )
    if n_ok < len(names):
        print(f"RADAR: resolved {n_ok} / {len(names)} rows to an audio file on disk")

    df = df.loc[audio_paths.notna()].copy()
    resolved_paths = audio_paths.loc[audio_paths.notna()]

    return pd.DataFrame(
        {
            "file_path": resolved_paths.map(lambda p: str(p.resolve())).values,
            "file": resolved_paths.map(lambda p: p.name).values,
            "file_stem": resolved_paths.map(lambda p: p.stem).values,
            "subgroup": df["source_site_csv"].values,
            "participant_id": df["participant_id"].values,
            "depressed": df["depressed"].values,
        }
    )


RADAR_IDX = build_wav_filename_index(RADAR_AUDIO_ROOT)
audio_df = prepare_radar_audio_df(RADAR_IDX)

print(f"Project root: {PROJECT}")
print(f"RADAR audio root: {RADAR_AUDIO_ROOT}")
print(f"Total recordings: {len(audio_df)}")
print("Recordings per site:")
print(audio_df["subgroup"].value_counts())
print(f"Labels — depressed=0: {(audio_df['depressed'] == 0).sum()} | depressed=1: {(audio_df['depressed'] == 1).sum()}")
print(f"Unique participants: {audio_df['participant_id'].nunique()}")

audio_df.head()

radar_model_dataset_raw_features-CIBER.csv: loaded 2086 rows (site_key='CIBER')
radar_model_dataset_raw_features-IISPV.csv: loaded 138 rows (site_key='IISPV')
radar_model_dataset_raw_features-KCL.csv: loaded 8515 rows (site_key='KCL')
radar_model_dataset_raw_features-VUmc.csv: loaded 3153 rows (site_key='VUmc')
RADAR: resolved 13884 / 13890 rows to an audio file on disk
Project root: /Users/k1777551/JansCode/ASMHI-Research-Project
RADAR audio root: /Users/k1777551/Library/CloudStorage/OneDrive-King'sCollegeLondon/RADAR-MDD
Total recordings: 13884
Recordings per site:
subgroup
KCL      8509
VUmc     3153
CIBER    2084
IISPV     138
Name: count, dtype: int64
Labels — depressed=0: 7676 | depressed=1: 6208
Unique participants: 497


,file_path,file,file_stem,subgroup,participant_id,depressed
0,/Users/k1777551/Library/CloudStorage/OneDrive-...,20200929_0800-scripted-1-1.wav,20200929_0800-scripted-1-1,CIBER,007751c5-d7ad-4bec-a58f-abf32500e2ae,1
1,/Users/k1777551/Library/CloudStorage/OneDrive-...,20200610_1100-scripted-2-1.wav,20200610_1100-scripted-2-1,CIBER,007751c5-d7ad-4bec-a58f-abf32500e2ae,1
2,/Users/k1777551/Library/CloudStorage/OneDrive-...,20200901_1100-unscripted-1.wav,20200901_1100-unscripted-1,CIBER,007751c5-d7ad-4bec-a58f-abf32500e2ae,1
3,/Users/k1777551/Library/CloudStorage/OneDrive-...,20210315_2000-scripted-1-1.wav,20210315_2000-scripted-1-1,CIBER,007751c5-d7ad-4bec-a58f-abf32500e2ae,1
4,/Users/k1777551/Library/CloudStorage/OneDrive-...,20210215_1100-scripted-2-1.wav,20210215_1100-scripted-2-1,CIBER,007751c5-d7ad-4bec-a58f-abf32500e2ae,1


In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import gc
import time
from functools import partial

import librosa
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import WhisperModel, WhisperProcessor

try:
    audio_df
except NameError as e:
    raise RuntimeError("Run the data-scan cell first (audio_df).") from e


# --------------------------------------------------
# Config
# --------------------------------------------------

WHISPER_ID = "openai/whisper-small.en"

TARGET_SR = 16_000
MAX_SECONDS = 30  # RADAR recordings may be longer than this; consider chunking if truncation loses too much signal

EPOCHS = 8
BATCH_SIZE = 8      # tune for your hardware — RADAR (~13.9k recordings) is far larger than Androids HC/PT
LEARNING_RATE = 1e-3
TEST_SIZE = 0.2     # swap for RADAR's official train/test split if one exists
RANDOM_STATE = 42
FREEZE_ENCODER = True
FORCE_CPU = False

RESULTS_PATH = PROJECT / "results" / "metrics" / "kintsugi_health" / "RADAR"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

torch.set_num_threads(min(8, max(1, os.cpu_count() or 1)))
# Apple Silicon (mps) checked before CUDA/CPU — the original CUDA-only check
# always fell back to CPU on this machine.
if not FORCE_CPU and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif not FORCE_CPU and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE} | recordings: {len(audio_df)} | freeze encoder: {FREEZE_ENCODER}")

# Site groupings: CIBER + IISPV are combined into one group -- IISPV alone is
# only 7 participants / 138 rows, too small to split into a meaningful
# train/test set on its own. Each group below trains its own model and gets
# its own summary/prediction CSVs.
SITE_GROUPS: dict[str, list[str]] = {
    "KCL": ["KCL"],
    "VUmc": ["VUmc"],
    "CIBER_IISPV": ["CIBER", "IISPV"],
}


# --------------------------------------------------
# Preprocessing
# --------------------------------------------------

def load_audio(path) -> np.ndarray:
    wav, _ = librosa.load(str(path), sr=TARGET_SR, mono=True)
    max_len = TARGET_SR * MAX_SECONDS
    if len(wav) > max_len:
        wav = wav[:max_len]
    return wav.astype(np.float32)


processor = WhisperProcessor.from_pretrained(WHISPER_ID)


# --------------------------------------------------
# Model (DAM-style: Whisper encoder + MLP head)
# --------------------------------------------------

class DAMLikeModel(nn.Module):
    """Whisper encoder + trainable head. Trained on RADAR's binary depressed label (PHQ-8 >= 10)."""

    def __init__(self, freeze_encoder: bool = True) -> None:
        super().__init__()
        self.whisper = WhisperModel.from_pretrained(WHISPER_ID, low_cpu_mem_usage=True)
        if freeze_encoder:
            for param in self.whisper.parameters():
                param.requires_grad = False

        hidden_size = self.whisper.config.d_model
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),  # binary depression; swap for regression if predicting PHQ-8 directly
        )

    def forward(self, input_features: torch.Tensor) -> torch.Tensor:
        encoder_out = self.whisper.encoder(input_features)
        pooled = encoder_out.last_hidden_state.mean(dim=1)
        return self.head(pooled).squeeze(-1)


# --------------------------------------------------
# Dataset
# --------------------------------------------------

class AudioDepressionDataset(Dataset):
    def __init__(self, paths: list[str], labels: np.ndarray) -> None:
        self.paths = paths
        self.labels = labels.astype(np.float32)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> tuple[np.ndarray, float]:
        return load_audio(self.paths[idx]), float(self.labels[idx])


def whisper_collate(batch: list[tuple[np.ndarray, float]], proc: WhisperProcessor):
    waves, labels = zip(*batch)
    # padding="max_length" (not padding=True) — Whisper requires a fixed
    # 3000-frame mel input; padding=True throws a length-mismatch error.
    feats = proc(list(waves), sampling_rate=TARGET_SR, padding="max_length", return_tensors="pt")
    y = torch.tensor(labels, dtype=torch.float32)
    return feats.input_features, y


def train_one_epoch(model, loader, optimizer, loss_fn) -> float:
    model.train()
    total_loss = 0.0
    n_batches = 0
    pbar = tqdm(loader, desc="Training batches", leave=False)
    for input_features, y in pbar:
        input_features = input_features.to(DEVICE)
        y = y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(input_features)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item())
        n_batches += 1
        pbar.set_postfix(avg_loss=f"{total_loss / n_batches:.4f}")
    return total_loss / max(n_batches, 1)


@torch.inference_mode()
def predict_probs(model, loader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_probs: list[np.ndarray] = []
    all_labels: list[np.ndarray] = []
    for input_features, y in tqdm(loader, desc="Evaluating", leave=False):
        input_features = input_features.to(DEVICE)
        logits = model(input_features)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


# --------------------------------------------------
# Per-group grouped train/test split, train + evaluate
# --------------------------------------------------

def run_group(group_name: str, sites: list[str]) -> None:
    group_df = audio_df[audio_df["subgroup"].isin(sites)].reset_index(drop=True)

    if group_df.empty:
        print(f"{group_name}: no rows found for sites {sites}, skipping")
        return

    print(f"\n=== {group_name} ({'+'.join(sites)}): {len(group_df)} rows, "
          f"{group_df['participant_id'].nunique()} participants ===")

    groups = group_df["participant_id"].astype(str)
    labels = group_df["depressed"].to_numpy(dtype=np.float32)
    paths = group_df["file_path"].tolist()

    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(group_df, labels, groups=groups))

    train_paths = [paths[i] for i in train_idx]
    test_paths = [paths[i] for i in test_idx]
    y_train = labels[train_idx]
    y_test = labels[test_idx]

    train_loader = DataLoader(
        AudioDepressionDataset(train_paths, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        collate_fn=partial(whisper_collate, proc=processor),
    )
    test_loader = DataLoader(
        AudioDepressionDataset(test_paths, y_test),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        collate_fn=partial(whisper_collate, proc=processor),
    )

    print(
        f"[{group_name}] Train: {len(train_paths)} rows "
        f"({group_df.iloc[train_idx]['participant_id'].nunique()} participants) | "
        f"Test: {len(test_paths)} rows ({group_df.iloc[test_idx]['participant_id'].nunique()} participants)"
    )
    print(f"[{group_name}] Train rows per site:")
    print(group_df.iloc[train_idx]["subgroup"].value_counts())
    print(f"[{group_name}] Test rows per site:")
    print(group_df.iloc[test_idx]["subgroup"].value_counts())

    model = DAMLikeModel(freeze_encoder=FREEZE_ENCODER).to(DEVICE)
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=LEARNING_RATE,
    )
    loss_fn = nn.BCEWithLogitsLoss()

    epoch_times: list[float] = []
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
        epoch_time = time.time() - t0
        epoch_times.append(epoch_time)

        avg_epoch_time = sum(epoch_times) / len(epoch_times)
        remaining_min = avg_epoch_time * (EPOCHS - epoch) / 60
        print(
            f"[{group_name}] Epoch {epoch}/{EPOCHS} train_loss={train_loss:.4f} "
            f"({epoch_time:.1f}s this epoch, ~{remaining_min:.1f} min remaining)"
        )

    test_probs, test_true = predict_probs(model, test_loader)
    test_pred = (test_probs >= 0.5).astype(int)

    metrics = {
        "model": f"DAM-like Whisper (RADAR - {group_name})",
        "sites": "+".join(sites),
        "n_total": len(group_df),
        "n_train": len(train_paths),
        "n_test": len(test_paths),
        "n_participants": group_df["participant_id"].nunique(),
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "freeze_encoder": FREEZE_ENCODER,
        "accuracy": accuracy_score(test_true, test_pred),
        "f1": f1_score(test_true, test_pred, zero_division=0),
        "roc_auc": roc_auc_score(test_true, test_probs) if len(np.unique(test_true)) > 1 else np.nan,
    }

    summary_df = pd.DataFrame([metrics])

    pred_df = group_df.iloc[test_idx][
        ["file_path", "file", "file_stem", "subgroup", "participant_id", "depressed"]
    ].copy()
    pred_df["pred_prob"] = test_probs
    pred_df["pred_label"] = test_pred

    summary_path = RESULTS_PATH / f"dam_whisper_radar_{group_name}_summary.csv"
    pred_path = RESULTS_PATH / f"dam_whisper_radar_{group_name}_test_predictions.csv"

    summary_df.to_csv(summary_path, index=False)
    pred_df.to_csv(pred_path, index=False)

    print(f"\n[{group_name}] Held-out test metrics:")
    print(summary_df.to_string(index=False))
    print(f"[{group_name}] Saved: {summary_path}")
    print(f"[{group_name}] Saved: {pred_path}")

    # Quick smoke inference on the first training file for this group
    sample_path = train_paths[0]
    sample_wav = load_audio(sample_path)
    sample_inputs = processor(sample_wav, sampling_rate=TARGET_SR, return_tensors="pt")
    with torch.inference_mode():
        sample_logit = model(sample_inputs.input_features.to(DEVICE))
        sample_prob = torch.sigmoid(sample_logit).item()
    print(f"[{group_name}] Smoke inference — {Path(sample_path).name}: P(depressed)={sample_prob:.3f}")

    model.cpu()
    del model, optimizer
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()


for group_name, sites in SITE_GROUPS.items():
    run_group(group_name, sites)

print("\nAll groups done. Outputs in:", RESULTS_PATH)


Device: mps | recordings: 13884 | freeze encoder: True


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

/var/folders/8b/z0f18krx04s7klg4yqm0fvs40000gr/T/ipykernel_19500/759799578.py:56: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, _ = librosa.load(str(path), sr=TARGET_SR, mono=True)
/opt/anaconda3/envs/jans_project/lib/python3.11/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Epoch 1/8 train_loss=0.6647
Epoch 2/8 train_loss=0.6379


KeyboardInterrupt: 